# SnapTEX phone-photo pilot
Run top to bottom in a Colab T4 GPU runtime. Download `phone-training-v0.2.zip` from the chat, then upload it in cell 2. The 20-page test split stays unused until the final cell. Do not put the private photos in the public GitHub repository.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, zipfile
repo = Path("/content/SnapTEX")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/Nikolay-Machev/SnapTEX.git", str(repo)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
assert (repo / "ml/prepare_phone_training.py").exists(), "Merge the phone-training PR before running this notebook."
uploaded = files.upload()
assert "phone-training-v0.2.zip" in uploaded
with zipfile.ZipFile("phone-training-v0.2.zip") as archive:
    archive.extractall(repo / "ml/data")
print("Dataset extracted.")

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], cwd=repo / "ml", check=True)
subprocess.run(["python", "validate_dataset.py", "--train", "data/phone-training-v0.2/train.jsonl", "--validation", "data/phone-training-v0.2/validation.jsonl", "--photo-test", "data/phone-training-v0.2/test.jsonl"], cwd=repo / "ml", check=True)

## Train
Two epochs, a frozen encoder, and no synthetic augmentation. The best validation epoch is saved locally. Keep this Colab runtime active until you export the checkpoint.


In [ ]:
subprocess.run(["python", "train.py", "--train", "data/phone-training-v0.2/train.jsonl", "--validation", "data/phone-training-v0.2/validation.jsonl", "--model", "tjoab/latex_finetuned", "--output", "/content/snaptex-phone-pilot-v0.2", "--epochs", "2", "--batch-size", "1", "--gradient-accumulation", "8", "--learning-rate", "1e-5", "--freeze-encoder", "--no-augment"], cwd=repo / "ml", check=True)
assert Path("/content/snaptex-phone-pilot-v0.2/model.safetensors").exists()

## Save the checkpoint privately
Add a Hugging Face **write** token to Colab Secrets as `HF_TOKEN`. This saves the finished checkpoint to a new private repository before the runtime ends. Keep the token out of code and output.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
secret = userdata.get("HF_TOKEN")
assert secret, "Add a Hugging Face write token named HF_TOKEN in Colab Secrets."
repo_id = "NikolayMachev/snaptex-phone-pilot-v0.2"
api = HfApi(token=secret)
api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
api.upload_folder(folder_path="/content/snaptex-phone-pilot-v0.2", repo_id=repo_id, ignore_patterns=["checkpoint-*", "runs/*"])
print("Private checkpoint saved:", repo_id)

## One final test
Only run this after training and validation choices are fixed. The report compares baseline and pilot on identical held-out crops. Invalid outputs count as failures.


In [ ]:
subprocess.run(["python", "evaluate_checkpoint.py", "--manifest", "data/phone-training-v0.2/test.jsonl", "--model", "tjoab/latex_finetuned", "--model", "/content/snaptex-phone-pilot-v0.2", "--output", "/content/phone-pilot-test.json"], cwd=repo / "ml", check=True)
files.download("/content/phone-pilot-test.json")

## Optional backup
The private Hugging Face repository above preserves the checkpoint. If you also want a Drive copy, mount Drive and copy the model folder after checking free space. Avoid running a second training session merely because `/content` was reset.
